# FinGPT Financial Reasoning v3: Program-Executed Answer

本 notebook 基于 `run_fingpt_v2.ipynb` 生成，但产物写入 `financial_reasoning_v3`，不覆盖 v2。v3 模型只训练 `Evidence + Program`，最终 `Normalized Answer` 由 executor 执行 Program 得到。



# 金融对话数值推理模型：SFT → 轻量 DPO→ benchmark 评估

目标：训练一个面向 **金融对话数值推理** 的 reasoning model。

整体闭环：
- **SFT-1（FinQA）**：先学习表文混合数值推理与程序监督
- **SFT-2（ConvFinQA）**：再学习多轮对话中的 follow-up 数值推理
- **DPO（可选）**：小规模优化表达质量、结构和少废话
- **GRPO（推荐）**：基于可验证 reward 优化答案正确性、程序一致性与结构约束
- **benchmark 评估**：`FinQA / ConvFinQA + CFLUE + FinanceBench + AdaptLLM/finance-tasks`

本 Notebook 复用 MedicalGPT 的训练框架：
- 数据格式遵循 `docs/datasets.md`
- pipeline 参考 `run_training_dpo_pipeline.ipynb`


 - v2 的风险：模型记住 Answer / Normalized Answer，学会心算或答案模板。
 - v3 的风险：模型过度拟合 Program DSL 和数据集里的 program pattern。

**v2 target 是：Evidence Program Answer Normalized Answer**
 - 其中 Normalized Answer 很容易变成“记答案”或“学数据集四舍五入模式”。尤其 FinQA 样本里的问题、表格和答案形式高度结构化，模型可能不是真的学会计算，而是在学：这个题型通常输出 0.xxxxx；百分比题输出小数 ratio；金额题输出某个裸数字

**v3 target 是：Evidence Program**
 - v3 会更容易过拟合到 Program 模式本身。
```text
percentage / growth / rate -> divide(subtract(a,b), b)
difference / change -> subtract(a,b)
portion / percent of total -> divide(a,b)
```
 - 如果过拟合，会表现为：看到关键词就套公式；只会输出 DSL，不会自然回答；Program 正确性对 executor 依赖很强（如果 executor 过宽松，模型可能学会半自然语言 Program，评估却被误判成功。我们已经看到 base 的一些输出有这个风险`subtract 25.14 from 60.94
`，所以 v3 的 overfit/shortcut 风险还包括“钻 executor 解析漏洞”。）
 - 模型不一定在训练集上死记硬背，但可能在学：怎样输出一个看起来像 Program、能骗过当前 executor 的字符串
 - 因为当前 executor 仍偏宽松，正式评估前需要收紧 strict DSL。否则 RL 阶段尤其危险，因为 reward 会奖励这些伪 Program。



**v3 是否比 v2 更容易在训练集上过拟合？**

就当前配置看：**不一定，甚至 v3 的答案记忆风险更低。**

原因：

- v3 target 更短，少了 `Answer / Normalized Answer`。
- v3 不训练模型心算最终答案。
- v3 数据量没有减少。
- v3 有独立 validation。
- SFT1 结果显示不是单纯记忆答案，而是 program execution 指标提升明显。

但 v3 更容易出现**能力窄化**：会写 Program；不会自然回答；会套公式；多轮状态容易混


**结论**

`run_fingpt_v3.ipynb` 相比 `run_fingpt_v2.ipynb`：

- **降低了答案记忆和小数心算过拟合风险**。
- **增加了 Program DSL 模式过拟合和任务场景窄化风险**。
- **当前最大风险是评估 executor 过宽松导致 metric overfitting**。
- **SFT1 的结果说明 v3 方向有效，但正式做 RL 前必须先收紧 strict DSL executor，并跑更大样本 + SFT2 对比。**

v3 不一定更容易过拟合，但更容易过拟合到“会写数据集 DSL / 套公式”，所以它适合作为可验证数值推理核心层，不适合作为唯一最终助手形态。

## 环境准备

如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [ ]:
import torch

# 查看 PyTorch 版本
print("PyTorch 版本:", torch.__version__)

# 查看 CUDA 版本（当前PyTorch使用的CUDA）
print("CUDA 版本:", torch.version.cuda)

# 查看是否启用GPU
print("CUDA 是否可用:", torch.cuda.is_available())


In [ ]:
%pip install -r requirements.txt --upgrade


In [ ]:
%pip install modelscope


In [ ]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


## 任务配置

这里把配置拆成五部分：
1. 阶段一数据 `SFT-1 (FinQA)`
2. 阶段二数据 `SFT-2 (ConvFinQA)`
3. 原始数据下载缓存目录
4. 转换 / 清洗 / 混合目录
5. 训练与评估输出目录


In [1]:
from pathlib import Path
import json
import random
from itertools import islice

BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct"

TEMPLATE_NAME = "qwen"
RANDOM_SEED = 42
FORCE_REDOWNLOAD_RAW = False

# ==== 数据处理主链配置 ====
# v3 使用独立 financial_reasoning_v3 根目录；模型只训练 Evidence + Program，答案由 executor 计算。
SFT_VARIANT = "program_executor_sft"
STRICT_TIERS = "A"
CONVFINQA_MODE = "turn_level"
FILTER_CONFLICTING_PROMPTS = True
SFT2_CONVFINQA_TO_FINQA_RATIO = 2.0
VALIDATION_CONVFINQA_ROWS = 307
VALIDATION_FINQA_ROWS = 153

# ==== 根路径统一配置（数据盘） ====
DISK_ROOT = Path("/root/autodl-tmp")
DATA_DIR = DISK_ROOT / "data" / "financial_reasoning_v3"
OUTPUT_ROOT = DISK_ROOT / "outputs" / "financial_reasoning_v3"

RAW_CACHE_DIR = DISK_ROOT / "data" / "financial_reasoning" / "raw"
SFT1_SFT_DIR = DATA_DIR / "sft1_sharegpt"
SFT2_SFT_DIR = DATA_DIR / "sft2_sharegpt"
DPO_DIR = DATA_DIR / "dpo_pairs"
NORMALIZED_DIR = DATA_DIR / "normalized"
ROUTER_AUDIT_DIR = DATA_DIR / "audit"
MIXED_DIR = DATA_DIR / "mixed"
CLEAN_DIR = DATA_DIR / "clean"
REPORT_DIR = DATA_DIR / "reports"
VALIDATION_DIR = DATA_DIR / "validation"

SFT1_MIXED_FILE = MIXED_DIR / "train_sft1_mixed.jsonl"
SFT2_MIXED_FILE = MIXED_DIR / "train_sft2_mixed.jsonl"
DPO_MIXED_FILE = DPO_DIR / "train_reasoning_dpo.jsonl"
SFT1_CLEAN_FILE = CLEAN_DIR / "train_sft1_clean.jsonl"
SFT2_CLEAN_FILE = CLEAN_DIR / "train_sft2_clean.jsonl"
SFT1_NORMALIZED_DIR = NORMALIZED_DIR / "sft1"
SFT2_NORMALIZED_DIR = NORMALIZED_DIR / "sft2"
SFT1_ROUTER_AUDIT_DIR = ROUTER_AUDIT_DIR / "sft1"
SFT2_ROUTER_AUDIT_DIR = ROUTER_AUDIT_DIR / "sft2"

SFT1_DIR = CLEAN_DIR / "sft1_dir_program"
SFT2_DIR = CLEAN_DIR / "sft2_dir_program"
VALIDATION_TRAIN_DIR = VALIDATION_DIR / "train_dir_program"
DPO_TRAIN_DIR = DPO_DIR / "train_dir"

SFT1_AUDIT_DIR = CLEAN_DIR / "audit_sft1"
SFT2_AUDIT_DIR = CLEAN_DIR / "audit_sft2"
SFT1_STRICT_FILE = CLEAN_DIR / "train_sft1_program_strict.jsonl"
SFT2_STRICT_FILE = CLEAN_DIR / "train_sft2_program_balanced.jsonl"
SFT_VALID_FILE = VALIDATION_DIR / "valid_program_balanced.jsonl"
SFT2_BALANCED_SUMMARY_FILE = CLEAN_DIR / "train_sft2_program_balanced_summary.json"
SFT2_CONVFINQA_ONLY_FILE = CLEAN_DIR / "train_sft2_convfinqa_turn_program_strict.jsonl"
SFT2_FINQA_REPLAY_FILE = CLEAN_DIR / "train_sft2_finqa_replay_program.jsonl"

# 训练/日志输出仍落在 v2 根目录，但目录名标识 dual answer。
SFT1_OUT = OUTPUT_ROOT / "sft1_program"
SFT1_MERGED_OUT = OUTPUT_ROOT / "sft1_program_merged"
SFT2_OUT = OUTPUT_ROOT / "sft2_program"
SFT2_MERGED_OUT = OUTPUT_ROOT / "sft2_program_merged"
DPO_OUT = OUTPUT_ROOT / "dpo"
DPO_MERGED_OUT = OUTPUT_ROOT / "dpo_merged"
TB_LOG_DIR = OUTPUT_ROOT / "tensorboard"

for d in [
    RAW_CACHE_DIR, SFT1_SFT_DIR, SFT2_SFT_DIR, DPO_DIR, NORMALIZED_DIR, ROUTER_AUDIT_DIR,
    SFT1_NORMALIZED_DIR, SFT2_NORMALIZED_DIR, SFT1_ROUTER_AUDIT_DIR, SFT2_ROUTER_AUDIT_DIR,
    MIXED_DIR, CLEAN_DIR, REPORT_DIR, VALIDATION_DIR,
    SFT1_DIR, SFT2_DIR, VALIDATION_TRAIN_DIR, DPO_TRAIN_DIR, SFT1_AUDIT_DIR, SFT2_AUDIT_DIR,
    OUTPUT_ROOT, TB_LOG_DIR, SFT1_OUT, SFT1_MERGED_OUT, SFT2_OUT, SFT2_MERGED_OUT, DPO_OUT, DPO_MERGED_OUT,
]:
    d.mkdir(parents=True, exist_ok=True)

SFT1_DATA_SPECS = [
    {
        "name": "finqa_train",
        "family": "finqa",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "finqa" / "train.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "target_rows": None,
        "sft_variant": SFT_VARIANT,
        "strict_tiers": STRICT_TIERS,
    },
]

SFT2_DATA_SPECS = [
    {
        "name": "convfinqa_train_turn",
        "family": "convfinqa_turn",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "convfinqa_turn" / "train_turn.json",
        "fallback_local_path": RAW_CACHE_DIR / "convfinqa" / "train.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "target_rows": None,
        "sft_variant": SFT_VARIANT,
        "strict_tiers": STRICT_TIERS,
        "convfinqa_mode": CONVFINQA_MODE,
    },
]

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS
print(json.dumps({
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "convfinqa_mode": CONVFINQA_MODE,
    "sft1_file": str(SFT1_STRICT_FILE),
    "sft2_file": str(SFT2_STRICT_FILE),
    "validation_file": str(SFT_VALID_FILE),
}, ensure_ascii=False, indent=2))


{
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v3",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v3",
  "sft_variant": "program_executor_sft",
  "convfinqa_mode": "turn_level",
  "sft1_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft1_program_strict.jsonl",
  "sft2_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v3/validation/valid_program_balanced.jsonl"
}


### 统一任务格式

本次两阶段 SFT 的主训练模板：
- **FinQA**：文本 + 表格 + 问题 -> 当前样本的 `Evidence / Program / Answer`
- **ConvFinQA**：历史状态 + 当前 turn 问题 + 表格/文本上下文 -> 当前 turn 的 `Evidence / Program / Answer`

ConvFinQA 的字段口径必须区分 current turn 与 final QA：
- 当前 turn question：优先 `annotation.cur_dial[-1]`，缺失时 fallback 到 `annotation.dialogue_break[turn_ind]`
- 当前 turn program：优先 `annotation.cur_program`
- 当前 turn answer：优先 `annotation.exe_ans`
- `qa.question / qa.program_re / qa.exe_ans` 是整题最终问题和最终 program，只能进入 raw metadata 或作为缺失字段 fallback，不能作为 turn-level 主监督

MedicalGPT 对数据格式的要求：
- **SFT**：`conversations`
- **DPO**：`question + response_chosen + response_rejected`

> 注：路由器仍支持 `fineval/fiqa_qa` family，但当前 notebook 的 SFT 两阶段不使用它们。


## 数据集处理流程



### 下载原始数据到本地缓存

这一阶段只负责下载 raw 数据，避免每次重跑 notebook 都重复下载。
- `url_json`：直接下载官方 JSON 文件
- `hf`：通过 `datasets.load_dataset` 落地到本地 jsonl
- 若缓存已存在且 `FORCE_REDOWNLOAD_RAW=False`，则直接跳过


In [2]:
import json
from datasets import load_dataset

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS


def raw_cache_file(spec: dict) -> Path:
    if spec["source_type"] == "local":
        return Path(spec["local_path"])
    return RAW_CACHE_DIR / spec["family"] / f"{spec['name']}.jsonl"

raw_files = {}
for spec in ALL_DATA_SPECS:
    cache_file = raw_cache_file(spec)
    raw_files[spec["name"]] = cache_file
    cache_file.parent.mkdir(parents=True, exist_ok=True)

    if spec["source_type"] == "local":
        if not cache_file.exists():
            raise FileNotFoundError(f"Missing local raw file: {cache_file}")
        print(f"[use local] {spec['name']} -> {cache_file}")
        continue

    if cache_file.exists() and not FORCE_REDOWNLOAD_RAW:
        print(f"[skip] use cached raw file: {cache_file}")
        continue

    print(f"[download] {spec['name']} -> {cache_file}")
    if spec["source_type"] == "hf":
        ds = load_dataset(spec["dataset_name"], split=spec["split"])
        with cache_file.open('w', encoding='utf-8') as f:
            for row in ds:
                f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
    else:
        raise ValueError(f"Unsupported source_type: {spec['source_type']}")
    print(f"[saved] {cache_file}")


[use local] finqa_train -> /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json
[use local] convfinqa_train_turn -> /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json


### 格式匹配

这一阶段只读取本地 raw 文件，并通过包入口执行当前数据处理主链：
- `python -m financial_data_processors --task sft`：生成 strict SFT，同时可选导出 `normalized / audit`
- `python -m financial_data_processors --task dpo`：基于 strict target 生成轻量 DPO pair

当前 SFT target 固定为英文三段式：`Evidence / Program / Answer`。


In [3]:

import json
import subprocess
import pandas as pd

sft_reports = []
dpo_reports = []

for spec in ALL_DATA_SPECS:
    raw_file = raw_files[spec['name']]
    is_sft1 = spec in SFT1_DATA_SPECS
    out_dir = SFT1_SFT_DIR if is_sft1 else SFT2_SFT_DIR
    norm_dir = SFT1_NORMALIZED_DIR if is_sft1 else SFT2_NORMALIZED_DIR
    audit_dir = SFT1_ROUTER_AUDIT_DIR if is_sft1 else SFT2_ROUTER_AUDIT_DIR

    sft_file = out_dir / f"{spec['name']}_sharegpt.jsonl"
    normalized_file = norm_dir / f"{spec['name']}_normalized.jsonl"
    audit_file = audit_dir / f"{spec['name']}_audit.jsonl"
    dpo_file = DPO_DIR / f"{spec['name']}_dpo.jsonl"

    common_router_args = [
        '--source_file', str(raw_file),
        '--dataset_family', spec['family'],
        '--sft_variant', spec.get('sft_variant', SFT_VARIANT),
        '--strict_tiers', spec.get('strict_tiers', STRICT_TIERS),
        '--filter_conflicting_prompts', 'true' if FILTER_CONFLICTING_PROMPTS else 'false',
    ]
    if spec['family'] == 'convfinqa_turn':
        common_router_args.extend(['--convfinqa_mode', spec.get('convfinqa_mode', CONVFINQA_MODE)])
    sharegpt_cmd = [
        'python', '-m', 'financial_data_processors',
        '--task', 'sft',
        '--output_file', str(sft_file),
        '--normalized_output_file', str(normalized_file),
        '--audit_output_file', str(audit_file),
        *common_router_args,
    ]
    dpo_cmd = [
        'python', '-m', 'financial_data_processors',
        '--task', 'dpo',
        '--output_file', str(dpo_file),
        '--seed', str(RANDOM_SEED),
        *common_router_args,
    ]

    print(' '.join(sharegpt_cmd))
    sharegpt_result = subprocess.run(sharegpt_cmd, check=True, capture_output=True, text=True)
    print(sharegpt_result.stdout)
    sft_reports.append(json.loads(sharegpt_result.stdout))

    print(' '.join(dpo_cmd))
    dpo_result = subprocess.run(dpo_cmd, check=True, capture_output=True, text=True)
    print(dpo_result.stdout)
    dpo_reports.append(json.loads(dpo_result.stdout))

print('[SFT conversion reports]')
print(json.dumps(sft_reports, ensure_ascii=False, indent=2))
print('[DPO conversion reports]')
print(json.dumps(dpo_reports, ensure_ascii=False, indent=2))

sft_summary_rows = []
for r in sft_reports:
    for fam, stats in r.get('per_family', {}).items():
        sft_summary_rows.append({
            'dataset_family': r.get('dataset_family', ''),
            'family': fam,
            'sft_variant': r.get('sft_variant'),
            'strict_tiers': r.get('strict_tiers'),
            'input_rows': stats.get('input_rows', 0),
            'strict_saved_rows': stats.get('saved_rows', 0),
            'normalized_rows': stats.get('normalized_rows', 0),
            'audit_rows': stats.get('audit_rows', 0),
            'tier_A_rows': stats.get('tier_A_rows', 0),
            'tier_B_rows': stats.get('tier_B_rows', 0),
            'tier_C_rows': stats.get('tier_C_rows', 0),
            'requires_history_rows': stats.get('requires_history_rows', 0),
            'multiturn_history_rows': stats.get('multiturn_history_rows', 0),
            'history_answer_missing_rows': stats.get('history_answer_missing_rows', 0),
            'history_full_reasoning_rows': stats.get('history_full_reasoning_rows', 0),
            'history_question_only_rows': stats.get('history_question_only_rows', 0),
            'history_full_reasoning_turns': stats.get('history_full_reasoning_turns', 0),
            'history_question_only_turns': stats.get('history_question_only_turns', 0),
            'rendered_history_full_reasoning_turns': stats.get('rendered_history_full_reasoning_turns', 0),
            'rendered_history_question_only_turns': stats.get('rendered_history_question_only_turns', 0),
            'duplicate_current_question_in_history_source_rows': stats.get('duplicate_current_question_in_history_source_rows', 0),
            'current_answer_leaked_in_history_source_rows': stats.get('current_answer_leaked_in_history_source_rows', 0),
            'question_semantic_risk_rows': stats.get('question_semantic_risk_rows', 0),
            'question_text_suspicious_rows': stats.get('question_text_suspicious_rows', 0),
            'weak_table_evidence_rendering_rows': stats.get('weak_table_evidence_rendering_rows', 0),
            'evidence_not_in_rendered_prompt_rows': stats.get('evidence_not_in_rendered_prompt_rows', 0),
            'evidence_visible_in_prompt_rows': stats.get('evidence_visible_in_prompt_rows', 0),
            'duplicate_current_question_in_history_rows': stats.get('duplicate_current_question_in_history_rows', 0),
            'current_answer_leaked_in_history_rows': stats.get('current_answer_leaked_in_history_rows', 0),
            'table_evidence_column_pruned_rows': stats.get('table_evidence_column_pruned_rows', 0),
            'exact_evidence_alignment_rows': stats.get('exact_evidence_alignment_rows', 0),
            'program_answer_match_rows': stats.get('program_answer_match_rows', 0),
            'raw_program_unchanged_rows': stats.get('raw_program_unchanged_rows', 0),
            'json_like_evidence_rows': stats.get('json_like_evidence_rows', 0),
        })

sft_summary_df = pd.DataFrame(sft_summary_rows)
print()
print('[SFT strict/normalized/audit summary table]')
display(sft_summary_df)

dpo_summary_rows = []
for r in dpo_reports:
    pf = r.get('dpo_post_filter', {})
    dpo_summary_rows.append({
        'dataset_family': r.get('dataset_family', ''),
        'input_rows': r.get('input_rows', 0),
        'saved_rows': r.get('saved_rows', 0),
        'skipped_rows': r.get('skipped_rows', 0),
        'post_filter_input_rows': pf.get('post_filter_input_rows', 0),
        'post_filter_saved_rows': pf.get('post_filter_saved_rows', 0),
        'post_filter_skipped_rows': pf.get('post_filter_skipped_rows', 0),
        'post_filter_duplicate_pair_rows': pf.get('post_filter_duplicate_pair_rows', 0),
        'post_filter_not_comparable_rows': pf.get('post_filter_not_comparable_rows', 0),
        'post_filter_rejected_too_short_rows': pf.get('post_filter_rejected_too_short_rows', 0),
    })

dpo_summary_df = pd.DataFrame(dpo_summary_rows)
print()
print('[DPO post-filter summary table]')
display(dpo_summary_df)


python -m financial_data_processors --task sft --output_file /root/autodl-tmp/data/financial_reasoning_v3/sft1_sharegpt/finqa_train_sharegpt.jsonl --normalized_output_file /root/autodl-tmp/data/financial_reasoning_v3/normalized/sft1/finqa_train_normalized.jsonl --audit_output_file /root/autodl-tmp/data/financial_reasoning_v3/audit/sft1/finqa_train_audit.jsonl --source_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json --dataset_family finqa --sft_variant program_executor_sft --strict_tiers A --filter_conflicting_prompts true
{
  "task": "sft",
  "output_file": "/root/autodl-tmp/data/financial_reasoning_v3/sft1_sharegpt/finqa_train_sharegpt.jsonl",
  "normalized_output_file": "/root/autodl-tmp/data/financial_reasoning_v3/normalized/sft1/finqa_train_normalized.jsonl",
  "audit_output_file": "/root/autodl-tmp/data/financial_reasoning_v3/audit/sft1/finqa_train_audit.jsonl",
  "dataset_family": "finqa",
  "sft_variant": "program_executor_sft",
  "strict_tiers": "A",
  "inpu

,dataset_family,family,sft_variant,strict_tiers,input_rows,strict_saved_rows,normalized_rows,audit_rows,tier_A_rows,tier_B_rows,...,weak_table_evidence_rendering_rows,evidence_not_in_rendered_prompt_rows,evidence_visible_in_prompt_rows,duplicate_current_question_in_history_rows,current_answer_leaked_in_history_rows,table_evidence_column_pruned_rows,exact_evidence_alignment_rows,program_answer_match_rows,raw_program_unchanged_rows,json_like_evidence_rows
0,finqa,finqa,program_executor_sft,A,6251,3694,6251,2557,3694,746,...,0,1395,4856,0,0,1773,6251,5703,6251,0
1,convfinqa_turn,convfinqa_turn,program_executor_sft,A,11104,6388,11104,4716,6388,2101,...,0,2432,8672,16,0,2393,11104,10825,11104,0



[DPO post-filter summary table]


,dataset_family,input_rows,saved_rows,skipped_rows,post_filter_input_rows,post_filter_saved_rows,post_filter_skipped_rows,post_filter_duplicate_pair_rows,post_filter_not_comparable_rows,post_filter_rejected_too_short_rows
0,finqa,6251,3667,2584,3694,3667,27,27,0,0
1,convfinqa_turn,11104,6374,4730,6388,6374,14,14,0,0


展示转换后的 `ConvFinQA / FinQA` SFT 样例，确认模板、表格上下文、历史对话和推理程序是否正常。

关键检查：

1. `FinQA` 是单轮表文推理：prompt 中使用 `Question:`，target 固定为 `Evidence / Program / Answer`。
2. `ConvFinQA` 是 turn-level multiturn：prompt 中使用 `Current question:`，该问题必须来自当前 turn，而不是 `qa.question` 的整题最终问题。
3. ConvFinQA history 优先渲染前序完整推理块：`Q + Evidence + Program + Answer`。拿不到可靠前序监督时才使用 question-only fallback。

ConvFinQA 正确样例形态：

```text
Conversation history:
Q: what is the net cash from operating activities in 2009?
Evidence:
- year ended june 30 , cash provided by operations increased $ 25587 to $ 206588 ...

Program: 206588
Answer: 206588

Current question: what about in 2008?

Respond exactly in this format:
Evidence:
- ...

Program: ...
Answer: ...
```

对应 target 应该是当前 turn，而不是最终 percentage change：

```text
Evidence:
- year ended june 30 , cash provided by operations increased $ 25587 to $ 206588 ...

Program: 181001
Answer: 181001
```

metadata 中应同时可见：
- `raw_metadata.current_question / current_program / current_exe_ans`
- `raw_metadata.final_question / final_program_re / final_exe_ans`

如果 `Current question` 等于 `final_question`，而当前 turn 不是最后一轮，说明数据处理又退化成 final-QA 监督，需要停止训练并修复。


### Strict 数据混合

`financial_data_processors` 已在转换阶段完成：
- `program_re` 执行校验
- evidence exact 对齐
- `answer_norm` 选择
- strict / normalized / audit 分流

因此这里不再执行旧的 `clean -> audit_sharegpt -> filter_sharegpt_by_audit` 主过滤流程，而是直接对 router 产出的 strict SFT 文件做抽样与合并，生成训练目录需要的 jsonl。


In [4]:
def sample_jsonl_records(path: Path, target_rows: int | None = None, seed: int = 42):
    with path.open('r', encoding='utf-8') as f:
        records = [line for line in f if line.strip()]
    if target_rows is None or target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]


def sample_records(records, target_rows: int | None = None, seed: int = 42):
    records = list(records)
    if target_rows is None or target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]


def read_jsonl_objects(path: Path):
    raw = path.read_text(encoding='utf-8')
    decoder = json.JSONDecoder()
    rows = []
    pos = 0
    bad_chunks = []
    literal_separator_count = 0

    while pos < len(raw):
        while pos < len(raw) and raw[pos].isspace():
            pos += 1
        while raw.startswith('\\n', pos):
            literal_separator_count += 1
            pos += 2
            while pos < len(raw) and raw[pos].isspace():
                pos += 1
        if pos >= len(raw):
            break
        try:
            obj, next_pos = decoder.raw_decode(raw, pos)
        except json.JSONDecodeError as e:
            bad_chunks.append({
                'char_pos': pos,
                'message': str(e),
                'snippet': raw[pos:pos + 160],
            })
            break
        rows.append(obj)
        pos = next_pos

    report = {
        'path': str(path),
        'rows': len(rows),
        'bad_chunks': len(bad_chunks),
        'literal_separator_count': literal_separator_count,
    }
    return rows, report, bad_chunks


def write_jsonl_objects(path: Path, rows):
    with path.open('w', encoding='utf-8') as wf:
        for row in rows:
            wf.write(json.dumps(row, ensure_ascii=False) + '\n')


In [5]:
def _record_prompt(row):
    conv = row.get("conversations") or []
    if conv and isinstance(conv[0], dict):
        return conv[0].get("value") or conv[0].get("content") or ""
    return row.get("prompt", "")


def _record_answer_norm(row):
    meta = row.get("metadata") or {}
    if meta.get("answer_norm") is not None:
        return str(meta.get("answer_norm"))
    conv = row.get("conversations") or []
    target = ""
    if len(conv) > 1 and isinstance(conv[1], dict):
        target = conv[1].get("value") or conv[1].get("content") or ""
    marker = "Normalized Answer:"
    if marker in target:
        return target.split(marker, 1)[1].strip().splitlines()[0].strip()
    marker = "Answer:"
    if marker in target:
        return target.split(marker, 1)[1].strip().splitlines()[0].strip()
    return ""


def filter_conflicting_prompt_labels(rows):
    prompt_to_norm = {}
    kept = []
    conflicts = 0
    for row in rows:
        prompt = _record_prompt(row)
        norm = _record_answer_norm(row)
        old = prompt_to_norm.get(prompt)
        if old is not None and old != norm:
            conflicts += 1
            continue
        prompt_to_norm[prompt] = norm
        kept.append(row)
    return kept, conflicts


# 1) SFT1 (FinQA) strict-A dual-answer 数据
with SFT1_STRICT_FILE.open("w", encoding="utf-8") as wf:
    for spec in SFT1_DATA_SPECS:
        path = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
        sampled = sample_jsonl_records(path, spec.get("target_rows"), seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith("\n") else line + "\n")

finqa_rows, _, bad = read_jsonl_objects(SFT1_STRICT_FILE)
if bad:
    raise ValueError(f"Invalid FinQA strict JSONL: {SFT1_STRICT_FILE} -> {bad[0]}")
finqa_rows, finqa_conflicts = filter_conflicting_prompt_labels(finqa_rows)
write_jsonl_objects(SFT1_STRICT_FILE, finqa_rows)
SFT1_MIXED_FILE.write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
SFT1_CLEAN_FILE.write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")

# 2) SFT2: ConvFinQA turn-level + FinQA replay, default ConvFinQA:FinQA = 2:1
convfinqa_rows = []
for spec in SFT2_DATA_SPECS:
    path = SFT2_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
    rows, _, bad = read_jsonl_objects(path)
    if bad:
        raise ValueError(f"Invalid SFT2 source JSONL: {path} -> {bad[0]}")
    convfinqa_rows.extend(rows)

convfinqa_rows, conv_conflicts = filter_conflicting_prompt_labels(convfinqa_rows)
random.Random(RANDOM_SEED).shuffle(convfinqa_rows)
random.Random(RANDOM_SEED + 1).shuffle(finqa_rows)

valid_conv_target = min(VALIDATION_CONVFINQA_ROWS, len(convfinqa_rows))
valid_finqa_target = min(VALIDATION_FINQA_ROWS, len(finqa_rows))
valid_conv = convfinqa_rows[:valid_conv_target]
valid_finqa = finqa_rows[:valid_finqa_target]
train_conv = convfinqa_rows[valid_conv_target:]
train_finqa_source = finqa_rows[valid_finqa_target:] or finqa_rows

finqa_train_target = int(round(len(train_conv) / SFT2_CONVFINQA_TO_FINQA_RATIO)) if SFT2_CONVFINQA_TO_FINQA_RATIO else len(train_finqa_source)
selected_replay = sample_records(train_finqa_source, finqa_train_target, seed=RANDOM_SEED + 2)
replayed_rows = max(0, len(selected_replay) - len(train_finqa_source))

sft2_rows = train_conv + selected_replay
valid_rows = valid_conv + valid_finqa
random.Random(RANDOM_SEED + 3).shuffle(sft2_rows)
random.Random(RANDOM_SEED + 4).shuffle(valid_rows)

write_jsonl_objects(SFT2_CONVFINQA_ONLY_FILE, train_conv)
write_jsonl_objects(SFT2_FINQA_REPLAY_FILE, selected_replay)
write_jsonl_objects(SFT2_STRICT_FILE, sft2_rows)
write_jsonl_objects(SFT_VALID_FILE, valid_rows)

SFT2_MIXED_FILE.write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
SFT2_CLEAN_FILE.write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")

# Make train_file_dir-style folders
(SFT1_DIR / SFT1_STRICT_FILE.name).write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
(SFT2_DIR / SFT2_STRICT_FILE.name).write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
(VALIDATION_TRAIN_DIR / SFT_VALID_FILE.name).write_text(SFT_VALID_FILE.read_text(encoding="utf-8"), encoding="utf-8")

def audit_program_executor_rows(rows):
    prompt_lengths = []
    target_answer_token_count = 0
    target_normalized_answer_token_count = 0
    const_token_count = 0
    question_in_first_512 = 0
    program_instruction_in_first_512 = 0
    program_execution_success = 0
    program_answer_match = 0
    for row in rows:
        conv = row.get("conversations") or []
        prompt = conv[0].get("value", "") if len(conv) > 0 else ""
        target = conv[1].get("value", "") if len(conv) > 1 else ""
        meta = row.get("metadata") or {}
        prompt_lengths.append(len(prompt))
        target_answer_token_count += target.count("Answer:")
        target_normalized_answer_token_count += target.count("Normalized Answer:")
        const_token_count += target.count("const_")
        question_in_first_512 += int(0 <= prompt.find("Current question:") < 512)
        program_instruction_in_first_512 += int(0 <= prompt.find("The final numeric answer will be computed by executing Program.") < 512)
        program_execution_success += int(meta.get("program_executable") is not None)
        program_answer_match += int(bool(meta.get("answer_matches_program")))
    prompt_lengths_sorted = sorted(prompt_lengths)
    def percentile(p):
        if not prompt_lengths_sorted:
            return 0
        idx = min(len(prompt_lengths_sorted) - 1, int(round((len(prompt_lengths_sorted) - 1) * p)))
        return prompt_lengths_sorted[idx]
    total = len(rows)
    return {
        "rows": total,
        "program_parse_success": round(program_execution_success / total, 6) if total else 0.0,
        "program_execution_success": round(program_execution_success / total, 6) if total else 0.0,
        "program_answer_match_rate": round(program_answer_match / total, 6) if total else 0.0,
        "const_token_count": const_token_count,
        "target_answer_token_count": target_answer_token_count,
        "target_normalized_answer_token_count": target_normalized_answer_token_count,
        "question_in_first_512_chars_rate": round(question_in_first_512 / total, 6) if total else 0.0,
        "program_instruction_in_first_512_chars_rate": round(program_instruction_in_first_512 / total, 6) if total else 0.0,
        "p50_prompt_chars": percentile(0.50),
        "p95_prompt_chars": percentile(0.95),
        "max_prompt_chars": max(prompt_lengths_sorted) if prompt_lengths_sorted else 0,
    }

summary = {
    "sft_variant": SFT_VARIANT,
    "convfinqa_mode": CONVFINQA_MODE,
    "same_prompt_conflicting_labels": finqa_conflicts + conv_conflicts,
    "sft1_rows": len(finqa_rows),
    "sft2_convfinqa_source_rows": len(convfinqa_rows),
    "sft2_train_convfinqa_rows": len(train_conv),
    "sft2_train_finqa_replay_rows": len(selected_replay),
    "sft2_train_rows": len(sft2_rows),
    "validation_convfinqa_rows": len(valid_conv),
    "validation_finqa_rows": len(valid_finqa),
    "validation_rows": len(valid_rows),
    "convfinqa_to_finqa_ratio": len(train_conv) / max(len(selected_replay), 1),
    "replayed_rows": replayed_rows,
    "sft2_convfinqa_only_file": str(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_file": str(SFT2_FINQA_REPLAY_FILE),
    "sft2_balanced_file": str(SFT2_STRICT_FILE),
    "validation_file": str(SFT_VALID_FILE),
    "train_audit": audit_program_executor_rows(sft2_rows),
    "validation_audit": audit_program_executor_rows(valid_rows),
    "sft1_audit": audit_program_executor_rows(finqa_rows),
}
SFT2_BALANCED_SUMMARY_FILE.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "sft_variant": "program_executor_sft",
  "convfinqa_mode": "turn_level",
  "same_prompt_conflicting_labels": 0,
  "sft1_rows": 3686,
  "sft2_convfinqa_source_rows": 6386,
  "sft2_train_convfinqa_rows": 6079,
  "sft2_train_finqa_replay_rows": 3040,
  "sft2_train_rows": 9119,
  "validation_convfinqa_rows": 307,
  "validation_finqa_rows": 153,
  "validation_rows": 460,
  "convfinqa_to_finqa_ratio": 1.9996710526315788,
  "replayed_rows": 0,
  "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_convfinqa_turn_program_strict.jsonl",
  "sft2_finqa_replay_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_finqa_replay_program.jsonl",
  "sft2_balanced_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v3/validation/valid_program_balanced.jsonl",
  "train_audit": {
    "rows": 9119,
    "program_parse_success": 1.0,
    "program

In [6]:
def line_count(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def _spec_report(spec):
    return {
        "name": spec.get("name"),
        "weight": spec.get("weight"),
        "max_rows": spec.get("max_rows"),
        "target_rows": spec.get("target_rows"),
        "sft_variant": spec.get("sft_variant", SFT_VARIANT),
        "strict_tiers": spec.get("strict_tiers", STRICT_TIERS),
        "convfinqa_mode": spec.get("convfinqa_mode"),
        "sampling_mode": "all_rows" if spec.get("target_rows") is None else "sampled",
    }

balanced_summary = {}
if SFT2_BALANCED_SUMMARY_FILE.exists():
    balanced_summary = json.loads(SFT2_BALANCED_SUMMARY_FILE.read_text(encoding="utf-8"))

report = {
    "disk_root": str(DISK_ROOT),
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "strict_tiers": STRICT_TIERS,
    "convfinqa_mode": CONVFINQA_MODE,
    "filter_conflicting_prompts": FILTER_CONFLICTING_PROMPTS,
    "target_schema": "Evidence / Program",
    "sft2_mixture": {
        "convfinqa_to_finqa_ratio": SFT2_CONVFINQA_TO_FINQA_RATIO,
        "balanced_summary": balanced_summary,
    },
    "sft1_allocations": [_spec_report(spec) for spec in SFT1_DATA_SPECS],
    "sft2_allocations": [_spec_report(spec) for spec in SFT2_DATA_SPECS],
    "sft1_raw_mix_rows": line_count(SFT1_MIXED_FILE),
    "sft1_clean_rows": line_count(SFT1_CLEAN_FILE),
    "sft1_strict_rows": line_count(SFT1_STRICT_FILE),
    "sft2_raw_mix_rows": line_count(SFT2_MIXED_FILE),
    "sft2_clean_rows": line_count(SFT2_CLEAN_FILE),
    "sft2_strict_rows": line_count(SFT2_STRICT_FILE),
    "sft2_convfinqa_only_rows": line_count(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_rows": line_count(SFT2_FINQA_REPLAY_FILE),
    "validation_rows": line_count(SFT_VALID_FILE),
}
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "disk_root": "/root/autodl-tmp",
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v3",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v3",
  "sft_variant": "program_executor_sft",
  "strict_tiers": "A",
  "convfinqa_mode": "turn_level",
  "filter_conflicting_prompts": true,
  "target_schema": "Evidence / Program",
  "sft2_mixture": {
    "convfinqa_to_finqa_ratio": 2.0,
    "balanced_summary": {
      "sft_variant": "program_executor_sft",
      "convfinqa_mode": "turn_level",
      "same_prompt_conflicting_labels": 0,
      "sft1_rows": 3686,
      "sft2_convfinqa_source_rows": 6386,
      "sft2_train_convfinqa_rows": 6079,
      "sft2_train_finqa_replay_rows": 3040,
      "sft2_train_rows": 9119,
      "validation_convfinqa_rows": 307,
      "validation_finqa_rows": 153,
      "validation_rows": 460,
      "convfinqa_to_finqa_ratio": 1.9996710526315788,
      "replayed_rows": 0,
      "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v

### 数据集配比说明（SFT v2）

当前策略：
- SFT-1：仅使用 **FinQA**，先学单轮表文混合数值推理
- SFT-2：使用 **ConvFinQA turn-level strict 样本 + FinQA replay**，再学多轮 follow-up 推理并保留单轮能力
- 如果未设置 `target_rows` 或总 budget，则默认使用该数据源转换后的全部 strict 样本
- SFT 联合训练：使用 `sft_dir_strict`，但必须先基于当前 strict 主链重建

本轮重点不是固定数据配比，而是 **target 设计与可验证性**：
- assistant target 固定为英文三段式：`Evidence / Program / Answer`
- evidence 使用 exact 对齐后的原始证据片段
- FinQA 的 `program_re` 来自原始 FinQA program
- ConvFinQA 的 `program_re` 表示当前 turn program，来源应是 `annotation.cur_program`；`qa.program_re` 是最终整题 program，只能保留为 final metadata
- 重新生成 strict 文件后，必须先检查 schema、evidence、program-answer match、current/final question 是否错位，再启动 SFT


In [7]:

def _norm_prompt_question(text):
    return " ".join("".join(ch.lower() if ch.isalnum() else " " for ch in str(text)).split())


def _prompt_current_question(prompt: str) -> str:
    marker = "Current question: "
    if marker not in prompt:
        return ""
    return prompt.split(marker, 1)[1].split("\n\nRespond", 1)[0].strip()


def _prompt_history_questions(prompt: str):
    questions = []
    question_only_mode = "Conversation history questions:" in prompt
    for line in prompt.splitlines():
        if line.startswith("Q: "):
            questions.append(line[3:].strip())
        elif question_only_mode and line.startswith("- "):
            questions.append(line[2:].strip())
    return questions


def audit_sft_jsonl(path: Path):
    rows = 0
    json_like_evidence = 0
    target_schema_ok = 0
    exact_evidence = 0
    program_answer_match = 0
    raw_program_present = 0
    raw_answer_mismatch = 0
    tier_counts = {'A': 0, 'B': 0, 'C': 0}
    requires_history = 0
    history_turn_rows = 0
    history_answer_missing = 0
    history_full_reasoning_rows = 0
    history_question_only_rows = 0
    history_full_reasoning_turns = 0
    history_question_only_turns = 0
    evidence_visible = 0
    evidence_not_in_prompt = 0
    duplicate_current_question = 0
    rendered_duplicate_current_question = 0
    current_answer_leaked = 0
    question_text_suspicious = 0
    table_evidence_column_pruned = 0
    convfinqa_current_question_metadata_mismatch = 0
    convfinqa_final_question_used_as_current = 0
    avg_prompt_chars = 0
    avg_answer_chars = 0
    first_examples = []

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            conv = row.get('conversations', [])
            prompt = conv[0]['value'] if len(conv) > 0 else ''
            answer = conv[1]['value'] if len(conv) > 1 else ''
            meta = row.get('metadata', {}) if isinstance(row.get('metadata'), dict) else {}
            raw_meta = meta.get('raw_metadata') if isinstance(meta.get('raw_metadata'), dict) else {}
            audit_flags = meta.get('audit_flags') or []
            semantic_flags = meta.get('semantic_audit_flags') or []

            avg_prompt_chars += len(prompt)
            avg_answer_chars += len(answer)
            if '{"text_' in answer or '{"table_' in answer:
                json_like_evidence += 1
            if all(anchor in answer for anchor in ['Evidence:', 'Program:']) and 'Answer:' not in answer and 'Normalized Answer:' not in answer:
                target_schema_ok += 1
            if meta.get('evidence_match_type') == 'exact':
                exact_evidence += 1
            if meta.get('answer_matches_program') is True:
                program_answer_match += 1
            if meta.get('program_raw'):
                raw_program_present += 1
            if 'raw_answer_mismatch_with_answer_norm' in audit_flags:
                raw_answer_mismatch += 1
            tier = str(meta.get('quality_tier') or 'A').upper()
            if tier in tier_counts:
                tier_counts[tier] += 1
            if meta.get('requires_history'):
                requires_history += 1
            if meta.get('history_turns', 0):
                history_turn_rows += 1
            if meta.get('history_answer_missing') is True:
                history_answer_missing += 1
            full_reasoning_turns = int(meta.get('history_full_reasoning_turns') or 0)
            question_only_turns = int(meta.get('history_question_only_turns') or 0)
            if full_reasoning_turns:
                history_full_reasoning_rows += 1
                history_full_reasoning_turns += full_reasoning_turns
            if question_only_turns:
                history_question_only_rows += 1
                history_question_only_turns += question_only_turns
            if meta.get('evidence_visible_in_prompt') is True:
                evidence_visible += 1
            if 'evidence_not_in_rendered_prompt' in audit_flags:
                evidence_not_in_prompt += 1
            if 'duplicate_current_question_in_history' in audit_flags:
                duplicate_current_question += 1
            if 'current_answer_leaked_in_history' in audit_flags:
                current_answer_leaked += 1

            prompt_current_question = _prompt_current_question(prompt)
            current_norm = _norm_prompt_question(prompt_current_question)
            if current_norm and any(_norm_prompt_question(q) == current_norm for q in _prompt_history_questions(prompt)):
                rendered_duplicate_current_question += 1
            raw_current_norm = _norm_prompt_question(raw_meta.get('current_question', ''))
            raw_final_norm = _norm_prompt_question(raw_meta.get('final_question', ''))
            if row.get('source_dataset') == 'ConvFinQA' and raw_current_norm and current_norm and current_norm != raw_current_norm:
                convfinqa_current_question_metadata_mismatch += 1
            if row.get('source_dataset') == 'ConvFinQA' and raw_final_norm and current_norm == raw_final_norm and raw_current_norm != raw_final_norm:
                convfinqa_final_question_used_as_current += 1

            if 'question_text_suspicious' in semantic_flags:
                question_text_suspicious += 1
            if meta.get('table_evidence_column_pruned') is True:
                table_evidence_column_pruned += 1
            if len(first_examples) < 2:
                first_examples.append({
                    'record_id': row.get('record_id', ''),
                    'conversation_id': row.get('conversation_id', ''),
                    'answer_preview': answer[:320],
                    'prompt_preview': prompt[:480],
                    'metadata_preview': {
                        'program_raw': meta.get('program_raw', ''),
                        'answer_norm': meta.get('answer_norm', ''),
                        'answer_display': meta.get('answer_display', ''),
                        'quality_tier': meta.get('quality_tier', ''),
                        'requires_history': meta.get('requires_history'),
                        'history_turns': meta.get('history_turns'),
                        'history_full_reasoning_turns': meta.get('history_full_reasoning_turns'),
                        'history_question_only_turns': meta.get('history_question_only_turns'),
                        'history_full_reasoning_ratio': meta.get('history_full_reasoning_ratio'),
                        'history_answer_missing': meta.get('history_answer_missing'),
                        'history_dependency_type': meta.get('history_dependency_type'),
                        'current_question': raw_meta.get('current_question', ''),
                        'current_program': raw_meta.get('current_program', ''),
                        'current_exe_ans': raw_meta.get('current_exe_ans'),
                        'final_question': raw_meta.get('final_question', ''),
                        'final_program_re': raw_meta.get('final_program_re', ''),
                        'final_exe_ans': raw_meta.get('final_exe_ans'),
                        'prompt_current_question': prompt_current_question,
                        'evidence_visible_in_prompt': meta.get('evidence_visible_in_prompt'),
                        'evidence_match_type': meta.get('evidence_match_type', ''),
                        'audit_flags': meta.get('audit_flags', []),
                        'semantic_audit_flags': meta.get('semantic_audit_flags', []),
                    },
                })

    return {
        'path': str(path),
        'rows': rows,
        'target_schema_ratio': round(target_schema_ok / rows, 6) if rows else 0.0,
        'json_like_evidence_ratio': round(json_like_evidence / rows, 6) if rows else 0.0,
        'exact_evidence_alignment_ratio': round(exact_evidence / rows, 6) if rows else 0.0,
        'program_answer_match_ratio': round(program_answer_match / rows, 6) if rows else 0.0,
        'raw_program_present_ratio': round(raw_program_present / rows, 6) if rows else 0.0,
        'raw_answer_mismatch_ratio': round(raw_answer_mismatch / rows, 6) if rows else 0.0,
        'tier_counts': tier_counts,
        'requires_history_ratio': round(requires_history / rows, 6) if rows else 0.0,
        'history_turn_rows_ratio': round(history_turn_rows / rows, 6) if rows else 0.0,
        'history_answer_missing_ratio': round(history_answer_missing / rows, 6) if rows else 0.0,
        'history_full_reasoning_rows_ratio': round(history_full_reasoning_rows / rows, 6) if rows else 0.0,
        'history_question_only_rows_ratio': round(history_question_only_rows / rows, 6) if rows else 0.0,
        'history_full_reasoning_turns': history_full_reasoning_turns,
        'history_question_only_turns': history_question_only_turns,
        'history_full_reasoning_turn_ratio': round(history_full_reasoning_turns / (history_full_reasoning_turns + history_question_only_turns), 6) if (history_full_reasoning_turns + history_question_only_turns) else 0.0,
        'evidence_visible_in_prompt_ratio': round(evidence_visible / rows, 6) if rows else 0.0,
        'evidence_not_in_prompt_ratio': round(evidence_not_in_prompt / rows, 6) if rows else 0.0,
        'duplicate_current_question_in_history_rows': duplicate_current_question,
        'rendered_duplicate_current_question_rows': rendered_duplicate_current_question,
        'current_answer_leaked_in_history_rows': current_answer_leaked,
        'convfinqa_current_question_metadata_mismatch_rows': convfinqa_current_question_metadata_mismatch,
        'convfinqa_final_question_used_as_current_rows': convfinqa_final_question_used_as_current,
        'convfinqa_current_question_metadata_mismatch_ratio': round(convfinqa_current_question_metadata_mismatch / rows, 6) if rows else 0.0,
        'convfinqa_final_question_used_as_current_ratio': round(convfinqa_final_question_used_as_current / rows, 6) if rows else 0.0,
        'question_text_suspicious_ratio': round(question_text_suspicious / rows, 6) if rows else 0.0,
        'table_evidence_column_pruned_rows': table_evidence_column_pruned,
        'avg_prompt_chars': round(avg_prompt_chars / rows, 2) if rows else 0.0,
        'avg_answer_chars': round(avg_answer_chars / rows, 2) if rows else 0.0,
        'examples': first_examples,
    }

audit_report = {
    'sft1_strict': audit_sft_jsonl(SFT1_STRICT_FILE),
    'sft2_strict': audit_sft_jsonl(SFT2_STRICT_FILE),
    'sft2_convfinqa_only': audit_sft_jsonl(SFT2_CONVFINQA_ONLY_FILE),
    'sft2_finqa_replay': audit_sft_jsonl(SFT2_FINQA_REPLAY_FILE),
}
print(json.dumps(audit_report, ensure_ascii=False, indent=2))


{
  "sft1_strict": {
    "path": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft1_program_strict.jsonl",
    "rows": 3686,
    "target_schema_ratio": 1.0,
    "json_like_evidence_ratio": 0.0,
    "exact_evidence_alignment_ratio": 1.0,
    "program_answer_match_ratio": 1.0,
    "raw_program_present_ratio": 1.0,
    "raw_answer_mismatch_ratio": 0.0,
    "tier_counts": {
      "A": 3686,
      "B": 0,
      "C": 0
    },
    "requires_history_ratio": 0.0,
    "history_turn_rows_ratio": 0.0,
    "history_answer_missing_ratio": 0.0,
    "history_full_reasoning_rows_ratio": 0.0,
    "history_question_only_rows_ratio": 0.0,
    "history_full_reasoning_turns": 0,
    "history_question_only_turns": 0,
    "history_full_reasoning_turn_ratio": 0.0,
    "evidence_visible_in_prompt_ratio": 1.0,
    "evidence_not_in_prompt_ratio": 0.0,
    "duplicate_current_question_in_history_rows": 0,
    "rendered_duplicate_current_question_rows": 0,
    "current_answer_leaked_in_history_rows": 0,


### SFT 数据重建建议

当前 notebook 使用独立 `financial_reasoning_v3` 根目录，但训练主链已更新为 QA 前置 + program-executed answer 格式。需要重新运行上面的 router 和 strict/balanced 混合单元，生成：

- `train_sft1_program_strict.jsonl`
- `train_sft2_convfinqa_turn_program_strict.jsonl`
- `train_sft2_program_balanced.jsonl`
- `valid_program_balanced.jsonl`
- `train_sft2_program_balanced_summary.json`
- `normalized/*.jsonl`
- `audit/*.jsonl`

重点关注：

- target 只包含 `Evidence:` 和 `Program:`，不包含 `Answer:` 或 `Normalized Answer:`
- metadata 中的 `answer_norm` 是否可解析，且由 `program_executable` 计算得到
- target 中是否还有 `const_100`、`const_7`、`const_<number>`
- `same_prompt_conflicting_labels` 是否为 `0`
- 当前问题、输出格式和 program execution instruction 是否位于 prompt 前 512 tokens
- SFT2 source ratio 是否接近 ConvFinQA:FinQA = 2:1
- ConvFinQA turn 分布、history dependency 分布是否写入 summary
- `evidence_visible_in_prompt_ratio` 应为 `1.0`
- `avg_prompt_chars` / `p95_prompt_chars` 是否可控


In [8]:
sft_manifest = {
    "entrypoint": "python -m financial_data_processors",
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "strict_tiers": STRICT_TIERS,
    "convfinqa_mode": CONVFINQA_MODE,
    "filter_conflicting_prompts": FILTER_CONFLICTING_PROMPTS,
    "sft1_strict_file": str(SFT1_STRICT_FILE),
    "sft2_strict_file": str(SFT2_STRICT_FILE),
    "sft2_convfinqa_only_file": str(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_file": str(SFT2_FINQA_REPLAY_FILE),
    "validation_file": str(SFT_VALID_FILE),
    "sft2_balanced_summary_file": str(SFT2_BALANCED_SUMMARY_FILE),
    "sft1_normalized_dir": str(SFT1_NORMALIZED_DIR),
    "sft2_normalized_dir": str(SFT2_NORMALIZED_DIR),
    "sft1_router_audit_dir": str(SFT1_ROUTER_AUDIT_DIR),
    "sft2_router_audit_dir": str(SFT2_ROUTER_AUDIT_DIR),
    "target_schema": "Evidence / Program",
    "prompt_policy": "QA + context + history; current question and output format first; history last.",
    "answer_policy": "Normalized Answer is computed by the executor from Program; model target is Evidence + Program only.",
    "sft2_mode": "convfinqa_mode=turn_level + FinQA replay balanced 2:1",
    "history_policy": "Build history from annotation.cur_dial/dialogue_break; keep history after context.",
    "note": "Do not modify supervised_finetuning truncation; rebuild strict dual files before starting SFT.",
}
print(json.dumps(sft_manifest, ensure_ascii=False, indent=2))


{
  "entrypoint": "python -m financial_data_processors",
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v3",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v3",
  "sft_variant": "program_executor_sft",
  "strict_tiers": "A",
  "convfinqa_mode": "turn_level",
  "filter_conflicting_prompts": true,
  "sft1_strict_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft1_program_strict.jsonl",
  "sft2_strict_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl",
  "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_convfinqa_turn_program_strict.jsonl",
  "sft2_finqa_replay_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_finqa_replay_program.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v3/validation/valid_program_balanced.jsonl",
  "sft2_balanced_summary_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_prog

## QuickEval: Base [done]


In [10]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/base_passk/


[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=base path=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct adapter=none
`torch_dtype` is deprecated! Use `dtype` instead!
Evaluating base: 100%|██████████████████████████| 16/16 [04:28<00:00, 16.80s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/base_passk


对比结论：**v3 的 base_passk 不能和 v2 的 base_passk 直接按 `answer_accuracy` 横向比较，因为评估目标已经变了。** v2 评的是“模型自己写的答案对不对”，v3 评的是“模型写的 Program 执行后对不对”。

但从结果看，base 在 v3 program-executed 口径下表现明显更差，说明 **base 本身不会稳定输出可执行 DSL Program**。

**整体指标对比**

| 指标 | v2 base | v3 base | 变化 |
| --- | ---: | ---: | ---: |
| macro primary_metric | 0.625 | 0.3125 | -0.3125 |
| macro pass@1_greedy | 0.625 | 0.3125 | -0.3125 |
| macro pass@1_sampled | 0.8125 | 0.25 | -0.5625 |
| macro pass@4 | 0.8125 | 0.375 | -0.4375 |
| macro pass@8 | 0.8125 | 0.5 | -0.3125 |
| avg prediction chars | 539.75 | 290.44 | 更短 |
| program_accuracy / string | 0.0625 | 0.25 | 提升 |

v3 新增的关键指标：

| v3 指标 | macro |
| --- | ---: |
| `program_parse_rate` | 1.0 |
| `program_execution_rate` | 0.875 |
| `executed_answer_accuracy` | 0.3125 |
| `model_normalized_answer_accuracy` | 0.125 |
| `program_answer_consistency` | 0.285714 |
| `program_string_accuracy` | 0.25 |

这里最关键的是：

```text
executed_answer_accuracy = 0.3125
model_normalized_answer_accuracy = 0.125
```

也就是说，**即使对 base 模型，用 executor 执行 Program 也比信任模型自己写答案更好**。但 base 的 Program 本身经常不规范，所以提升有限。

**按任务看**

| task | v2 answer acc | v3 executed acc | v3 program execution |
| --- | ---: | ---: | ---: |
| ConvFinQA | 0.75 | 0.375 | 1.0 |
| FinQA | 0.50 | 0.25 | 0.75 |
| Macro | 0.625 | 0.3125 | 0.875 |

v3 ConvFinQA 的 `program_execution_rate=1.0` 看起来很好，但要小心，下面会说这是有水分的。

**逐题观察**

v2 base 经常这样输出：

```text
Program:
1. Extract ...
2. Calculate ...

Answer: ...
Normalized Answer: ...
```

它的 `Program` 不是 DSL，但它能直接写出答案，所以 v2 口径下得分高。

v3 base 被要求只输出 Program 后，出现了很多非规范 Program：

```text
Program: subtract 25.14 from 60.94
```

```text
Program: divide (8.1, 56.0) * 100
```

```text
Program: get_equipment_rents_payable_2008 = equipment rents payable at dec. 31 2008 from table
```

```text
Program: ```python
rent_2016 = 317
...
```

这些都说明：**base 没有学会我们需要的 canonical DSL。**

但是它也有一些正确例子：

```text
Program: subtract(5829, 5735)
executed_answer: 94
gold: 94
```

```text
Program: divide(463, 4612)
executed_answer: 0.10039
gold: 0.10039
```

所以 v3 评估方向是对的：它能识别“模型写出了可执行程序，因此系统能算对答案”。

**一个重要问题：v3 当前 executor 太宽松**

我发现 v3 评估里有个需要修的点：`execute_program` 当前对非 DSL 文本过于宽松。

例如：

```text
Program: subtract 25.14 from 60.94
```

当前会被解析出第一个数字 `25.14`，于是被认为 execution success。

又比如：

```text
Program: get_equipment_rents_payable_2008 = ...
```

当前可能解析出 `2008`，也算 execution success。

所以：

```text
program_execution_rate = 0.875
```

这个指标现在是虚高的。

我做了一个粗略 strict DSL 检查：

```text
v3 greedy total: 16
strict-looking DSL programs: 10 / 16 = 0.625
executor_success_but_not_strict_DSL: 5
```

按任务：

| task | strict-looking DSL | executor success but not strict |
| --- | ---: | ---: |
| ConvFinQA | 6/8 | 2 |
| FinQA | 4/8 | 3 |

所以真实情况更接近：

```text
base 能输出严格 DSL 的比例约 62.5%
当前 program_execution_rate 0.875 偏乐观
```

**这次对比说明什么**

1. **v2 base 得分高，是因为 base 擅长直接自然语言算答案**
   它不擅长生成 canonical Program，但能直接写 `Normalized Answer`。

2. **v3 base 得分低，是预期的**
   base 没经过 program-only SFT，不应该指望它稳定输出 `divide(...) / subtract(...)` 这类 DSL。

3. **v3 指标更适合衡量我们真正想训练的能力**
   v3 看的是“模型能不能写出可执行计算程序”，不是“模型能不能心算小数”。

4. **必须先修 executor 严格性**
   否则 `program_execution_rate` 会被自然语言 Program 里的数字误判拉高。

**建议下一步**

优先改评估执行器的入口规则：

```text
只有以下两类 Program 可以执行：
1. 纯数字 literal，例如 25.14
2. canonical DSL call，例如 divide(8.1, 56.0)、subtract(93, 103)
```

不允许这些被执行成功：

```text
subtract 25.14 from 60.94
divide (8.1, 56.0) * 100
get_equipment_rents_payable_2008 = ...
```python ...
```

修完后重新跑：

```text
v3 base_passk
v3 sft1_program_passk
```

预期会看到：

```text
base program_execution_rate 下降
sft1_program program_execution_rate 明显高于 base
```

那才是 v3 路线真正有说服力的对比。

一句话总结：

**v2 base 看起来强，是因为它直接答题；v3 base 变弱，是因为 v3 要求它生成可执行 Program，而 base 没训练过这个能力。当前 v3 评估方向正确，但 executor 需要先收紧，否则 program_execution_rate 会被自然语言 Program 误判虚高。**

## SFT

### SFT-1：FinQA 主干推理训练

对应当前设置：
1. 阶段一数据集为 `FinQA`
2. 已展示原始样本与转换后样本
3. 已完成清洗与严格过滤
4. 已统一转换为 MedicalGPT SFT 格式
5. 这里给出 `SFT-1` 训练命令


In [9]:
BASE_MODEL, SFT1_DIR, str(SFT1_OUT)


('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct',
 PosixPath('/root/autodl-tmp/data/financial_reasoning_v3/clean/sft1_dir_program'),
 '/root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program')

In [ ]:
sft1_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', BASE_MODEL,
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT1_DIR),

    '--validation_split_percentage', '1',
    '--do_eval',
    '--eval_steps', '100',
    '--eval_strategy', 'steps',

    '--do_train',
    '--use_peft',

    '--num_train_epochs', '2',

    '--per_device_train_batch_size', '1',
    '--max_grad_norm', '1.0',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',

    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',
    '--learning_rate', '1e-5',

    '--logging_steps', '10',
    '--save_steps', '200',

    '--logging_first_step', 'True',
    '--report_to', 'tensorboard',
    '--logging_dir', str(TB_LOG_DIR / 'sft1'),

    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'bfloat16',

    '--device_map', 'auto',
    
    '--output_dir', str(SFT1_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft1_cmd))


In [ ]:
!python -m training.supervised_finetuning --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v3/clean/sft1_dir_program \
    --validation_split_percentage 1 \
    --do_eval --eval_steps 100 --eval_strategy steps --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 1.0 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --warmup_steps 30 --weight_decay 0.05 --learning_rate 5e-6 \
    --logging_steps 10 --save_steps 200 --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v3/tensorboard/sft1_program \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program \
    --template_name qwen \
    --preprocessing_num_workers 16


In [11]:
!python -m tooling.merge_peft_adapter \
    --base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --lora_model /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged


Namespace(base_model='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='/root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program', resize_emb=False, output_dir='/root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct
LoRA model: /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program
Loading LoRA for causal language model
Loading weights: 100%|████████████████████████| 339/339 [00:03<00:00, 98.15it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100%|███████████████████████| 2/2 [00:30<00:00, 15.31s/it]
Done! model saved to /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged


### SFT1 QuickEval

基于已经 merge 的 `SFT1_MERGED_OUT` 做快速 benchmark，先回答一个问题：

> SFT-1（FinQA）训练后，模型是否在 FinQA / ConvFinQA quick benchmark 上不低于 base？


In [12]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry sft1=/root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft1_program_passk/


[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=sft1 path=/root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged adapter=none
`torch_dtype` is deprecated! Use `dtype` instead!
Evaluating sft1: 100%|██████████████████████████| 16/16 [04:04<00:00, 15.30s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft1_program_passk


结论：**v3 SFT1 是有效的，而且提升很明显；但是否值得立刻做 RL，答案是“值得准备，但先修评估 executor，再跑更大样本和 SFT2”。**

当前 8+8 quick benchmark 只能做 smoke test，不过信号很清楚。

**整体对比**

| 指标 | base | sft1_program | 提升 |
| --- | ---: | ---: | ---: |
| macro `executed_answer_accuracy` / primary | 0.3125 | 0.6875 | +0.375 |
| macro `pass@1_greedy` | 0.3125 | 0.6875 | +0.375 |
| macro `pass@4` | 0.375 | 0.8125 | +0.4375 |
| macro `pass@8` | 0.5 | 0.875 | +0.375 |
| macro `program_string_accuracy` | 0.25 | 0.5625 | +0.3125 |
| macro `program_execution_rate` | 0.875 | 1.0 | +0.125 |

按任务看：

| task | base pass@1 | sft1 pass@1 | base pass@8 | sft1 pass@8 |
| --- | ---: | ---: | ---: | ---: |
| FinQA | 0.25 | 0.875 | 0.5 | 1.0 |
| ConvFinQA | 0.375 | 0.5 | 0.5 | 0.75 |
| Macro | 0.3125 | 0.6875 | 0.5 | 0.875 |

这说明 SFT1 对 FinQA 的 program learning 很有效：

```text
FinQA pass@1: 0.25 -> 0.875
FinQA pass@8: 0.5 -> 1.0
```

ConvFinQA 也有提升，但没有 FinQA 那么强：

```text
ConvFinQA pass@1: 0.375 -> 0.5
ConvFinQA pass@8: 0.5 -> 0.75
```

这符合预期，因为 SFT1 只训 FinQA，ConvFinQA 应该主要看 SFT2。

**逐题信号**

FinQA 上 SFT1 改善非常明显：

- `FIS/2010/page_70.pdf-2`
  - base: 变量式 program，不可执行
  - sft1: `divide(121.4, 4187.8)`，正确
  - sampled correct: `0/8 -> 8/8`

- `GS/2015/page_188.pdf-2`
  - base: 输出 Python block，不可执行
  - sft1: `divide(301, 2575)`，正确
  - sampled correct: `0/8 -> 8/8`

- `STT/2007/page_111.pdf-3`
  - base: 自然语言 program，被错误解析
  - sft1: `divide(subtract(4711, 4926), 4926)`，正确
  - sampled correct: `0/8 -> 6/8`

- `CMCSA/2015/page_112.pdf-2`
  - base: `subtract 1171 from 1136`，非 DSL
  - sft1: `subtract(1136, 1171)`，正确
  - sampled correct: `2/8 -> 6/8`

这些都说明 SFT1 学到了核心能力：**把自然语言金融问题编译成可执行 DSL Program**。

**但也有问题**

1. **百分比/ratio 仍有单位尺度错误**

`INTC/2015/page_41.pdf-4`：

```text
gold: divide(8.1, 56.0) = 0.14464
sft1: divide(multiply(8.1, 100), 56.0) = 14.464286
```

模型把 ratio 和 percent 混了。这个是 v3 里最需要 reward/数据规则强化的点。

2. **ConvFinQA history 仍不稳**

例如：

`Single_MRO/...-1_3` 问的是：

```text
what was the weighted average exercise price per share in 2005?
```

gold 是：

```text
25.14
```

但 sft1 输出：

```text
divide(subtract(60.94, 25.14), 25.14)
```

这是把后续百分比变化题的程序套过来了，说明 SFT1 没学好多轮 turn state。这个要靠 SFT2 解决，不建议只靠 RL 硬推。

3. **executor 仍然偏宽松**

之前已经看到，base 的一些自然语言 program 会被错误当作数字执行成功，例如：

```text
subtract 25.14 from 60.94
```

可能被解析成 `25.14`。所以现在的 `program_execution_rate` 有虚高风险。正式做 RL 前必须修 strict executor，否则 reward 会被污染。

**pass@k 是否说明值得做 RL？**

值得，但有前提。

SFT1 当前 macro：

```text
pass@1 = 0.6875
pass@8 = 0.875
gap = 0.1875
```

FinQA：

```text
pass@1 = 0.875
pass@8 = 1.0
gap = 0.125
```

ConvFinQA：

```text
pass@1 = 0.5
pass@8 = 0.75
gap = 0.25
```

这说明：**模型采样空间里已经存在更多正确 program，尤其 ConvFinQA 还有较大 pass@k - pass@1 空间。** 这正是 RL / verifier-guided optimization 有价值的条件。

但是注意：

```text
pass@1_sampled = 0.5
pass@1_greedy = 0.6875
```

采样第一个样本反而比 greedy 差，说明当前模型分布还不够稳定。RL 的目标应该是把 pass@k 里的正确 Program 往 greedy / high-probability 区域推，而不是只提高采样多样性。

**是否现在就做 RL？**

我的建议是分三步：

**第一步：先修评估 executor**

必须先做，否则 RL reward 不可靠。

要求：

```text
只允许纯数字 literal
只允许 canonical DSL call
只允许多步 DSL + #引用
禁止自然语言 / Python block / 变量赋值 / 中缀表达式被执行成功
```

否则 RL 会奖励一些伪 program。

**第二步：先跑 SFT2 program**

现在 SFT1 已经证明 FinQA program learning 有效。下一步应该训练/评估：

```text
sft2_program_merged
```

重点看：

```text
ConvFinQA pass@1 是否从 0.5 继续上升
FinQA 是否不明显掉
```

如果 SFT2 后 ConvFinQA 仍有明显 `pass@8 > pass@1`，RL 更值得做。

**第三步：做 RL / GRPO**

值得做的 reward 主项：

```text
execute(model_program) == gold_answer_norm
```

辅助 reward：

```text
program_parse_reward
strict_dsl_reward
program_execution_reward
executed_answer_correct_reward
ratio_percent_scale_reward
evidence_format_reward
```

不要把模型自己写的 `Normalized Answer` 当主 reward，因为 v3 的核心就是不让模型心算。

**当前判断**

SFT 是否有效：

```text
是，有明显提升。
```

尤其 FinQA：

```text
executed_answer_accuracy: 0.25 -> 0.875
pass@8: 0.5 -> 1.0
```

是否值得做 RL：

```text
值得，但不要马上基于当前 executor 做。
```

推荐顺序：

```text
1. 收紧 strict executor
2. 重跑 base_passk / sft1_program_passk
3. 训练并评估 sft2_program_passk
4. 如果 pass@8 > pass@1 仍明显，做 GRPO/RL
```

一句话总结：

**SFT1 已经有效地把模型从“自然语言答题”推向“可执行 Program 生成”；pass@k 显示还有可优化空间，尤其 ConvFinQA 值得做 RL。但 RL 前必须先修 strict program executor，否则 reward 会奖励假可执行程序。**

### SFT-2：ConvFinQA 对话推理强化

在 `SFT-1` 完成后：
- 先 merge `SFT-1 LoRA`
- 再用 `ConvFinQA` 做二阶段 SFT（强化多轮 follow-up 推理）


In [13]:
str(SFT2_DIR), str(SFT2_OUT), str(TB_LOG_DIR / 'sft2')


('/root/autodl-tmp/data/financial_reasoning_v3/clean/sft2_dir_program',
 '/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program',
 '/root/autodl-tmp/outputs/financial_reasoning_v3/tensorboard/sft2')

In [ ]:
# sft1
!python -m training.supervised_finetuning --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v3/clean/sft1_dir_program \
    --validation_split_percentage 1 \
    --do_eval --eval_steps 100 --eval_strategy steps --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 1.0 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --warmup_steps 30 --weight_decay 0.05 --learning_rate 5e-6 \
    --logging_steps 10 --save_steps 200 --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v3/tensorboard/sft1_program \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program \
    --template_name qwen \
    --preprocessing_num_workers 16


In [ ]:
# sft2
!python -m training.supervised_finetuning \
    --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning_v3/sft1_program_merged \
    --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v3/clean/sft2_dir_program \
    --validation_split_percentage 1 --do_eval --eval_steps 100 --eval_strategy steps \
    --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft \
    --num_train_epochs 2 --per_device_train_batch_size 1 \
    --max_grad_norm 0.5 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --learning_rate 5e-6 --warmup_steps 50 --weight_decay 0.05 \
    --logging_steps 10 --save_steps 200 --save_total_limit 2 \
    --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v3/tensorboard/sft2_program \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --device_map auto \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program \
    --preprocessing_num_workers 16 \
    --template_name qwen


In [1]:
!python -m tooling.merge_peft_adapter --base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --lora_model /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged



libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
Namespace(base_model='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program', resize_emb=False, output_dir='/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct
LoRA model: /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program
Loading LoRA for causal language model
Loading weights: 100%|████████████████████████| 339/339 [00:09<00:00, 36.87it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100%|███████████████████████| 2/2 [00:26<00:00, 13.40s/it]
Done! model saved to /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged


### SFT2 QuickEval

基于已经 merge 的 `SFT2_MERGED_OUT` 做快速 benchmark

做pass@k & pass@1


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct\
  --model_entry sft=/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --finqa_max_samples 16 \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --convfinqa_max_samples 16\
  --max_new_tokens 1024 \
  --temperature 0.0 \
  --processor_sft_variant program_executor_sft \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/benchmark_quick_program


In [15]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry sft2=/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged\
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft2_merged_passk/


[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=sft2 path=/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged adapter=none
`torch_dtype` is deprecated! Use `dtype` instead!
Evaluating sft2: 100%|██████████████████████████| 16/16 [02:53<00:00, 10.85s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft2_merged_passk


### quickeval summary

> 每组都是 `num_examples=16`、`pass_k=[1,4,8]`、每题采样 8 次。注意样本很小，`0.0625` 就是 1 道题的差异。

**v3 三组对比**

| v3 模型 | primary / answer_acc | pass@1 greedy | pass@1 sampled | program_acc | executed_acc | pass@4 | pass@8 | avg chars |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| base_passk | 0.3125 | 0.3125 | 0.2500 | 0.2500 | 0.3125 | 0.3750 | 0.5000 | 290.4 |
| sft1_program_passk | 0.6875 | 0.6875 | 0.5000 | 0.5625 | 0.6875 | 0.8125 | 0.8750 | 230.4 |
| sft2_merged_passk | **0.8750** | **0.8750** | **0.8125** | **0.7500** | **0.8750** | **0.9375** | **0.9375** | **144.8** |

结论：v3 里 `sft2_merged_passk` 最强。相对 base，primary 从 `0.3125 -> 0.8750`，提升 `+0.5625`，也就是 16 题里多对 9 题。`sft1_program` 已经明显有效，但 `sft2` 又在 `sft1` 基础上多提升 `+0.1875`，也就是 3 题。

按任务拆开看：

| v3 模型 | ConvFinQA primary | FinQA primary | ConvFinQA pass@8 | FinQA pass@8 |
|---|---:|---:|---:|---:|
| base | 0.3750 | 0.2500 | 0.5000 | 0.5000 |
| sft1_program | 0.5000 | **0.8750** | 0.7500 | **1.0000** |
| sft2_merged | **0.8750** | **0.8750** | **0.8750** | **1.0000** |

这里很清楚：`sft1_program` 主要把 FinQA 拉起来了；`sft2_merged` 进一步把 ConvFinQA 从 `0.5` 拉到 `0.875`，所以整体最稳。

**v2 / v3 总比较**

| 版本/模型 | primary | pass@1 greedy | pass@1 sampled | program_acc | pass@4 | pass@8 | avg chars |
|---|---:|---:|---:|---:|---:|---:|---:|
| v2 base_passk | 0.6250 | 0.6250 | 0.8125 | 0.0625 | 0.8125 | 0.8125 | 539.8 |
| v2 sft1_dual_passk | 0.2500 | 0.2500 | 0.3750 | 0.3750 | 0.5625 | 0.5625 | 234.1 |
| v2 sft2_merged_passk | 0.8125 | 0.8125 | 0.8125 | **0.8125** | **0.9375** | **0.9375** | 268.3 |
| v2 dpo_passk | 0.8125 | 0.8125 | 0.7500 | **0.8125** | **0.9375** | **0.9375** | 269.0 |
| v3 base_passk | 0.3125 | 0.3125 | 0.2500 | 0.2500 | 0.3750 | 0.5000 | 290.4 |
| v3 sft1_program_passk | 0.6875 | 0.6875 | 0.5000 | 0.5625 | 0.8125 | 0.8750 | 230.4 |
| v3 sft2_merged_passk | **0.8750** | **0.8750** | **0.8125** | 0.7500 | **0.9375** | **0.9375** | **144.8** |

主要结论：

1. **全局最佳是 v3 `sft2_merged_passk`**  
   primary `0.8750`，比 v2 最好的 `sft2/dpo` 的 `0.8125` 高 `+0.0625`，也就是多 1 题。pass@4/pass@8 和 v2 最强持平，都是 `0.9375`。

2. **v3 的训练链路比 v2 更健康**  
   v2 是 `base 0.625 -> sft1 0.25 -> sft2 0.8125`，中间 `sft1_dual` 明显退化。  
   v3 是 `base 0.3125 -> sft1_program 0.6875 -> sft2 0.875`，每一步都向上走。

3. **v3 sft2 更短、更准，但 program 指标略低于 v2 sft2**  
   v3 sft2 的 average prediction chars 是 `144.8`，v2 sft2 是 `268.3`，明显更简洁。  
   但 `program_accuracy` 是 v3 `0.7500` vs v2 `0.8125`，`program_answer_consistency` 是 v3 `0.3125` vs v2 `0.5`。也就是说 v3 最终答案更准，但程序字符串/程序一致性不如 v2 sft2。

4. **v2 DPO 没有带来主指标提升**  
   v2 `dpo_passk` 和 v2 `sft2_merged_passk` primary 都是 `0.8125`，pass@4/pass@8 也都 `0.9375`；DPO 的 `pass@1_sampled` 还从 `0.8125` 降到 `0.7500`。这组结果里 DPO 基本没赚到。

5. **base 的 v2/v3 不建议直接当模型能力比较**  
   两边都是同一个 base 模型，但 v3 manifest 使用了 `processor_sft_variant=program_executor_sft` 和 `primary_metric=executed_answer_accuracy`，v2 base manifest 没有这些字段；所以 v3 base 从 `0.625` 掉到 `0.3125` 更像是评测/输出格式约束变化导致的，不一定代表模型本体变差。

> `v3 sft2_merged_passk` > `v2 sft2_merged_passk ≈ v2 dpo_passk` > `v3 sft1_program_passk` > `v2 base_passk` > `v3 base_passk` > `v2 sft1_dual_passk`。

## DPO

轻量 DPO 目标：
- 优化表达质量
- 保持结构完整
- 减少废话
- 不让偏好训练覆盖主干 reasoning 能力


In [16]:
all_specs = SFT1_DATA_SPECS + SFT2_DATA_SPECS
DPO_TOTAL_BUDGET_VALUE = globals().get('DPO_TOTAL_BUDGET')
MAX_DPO_PER_DATASET_VALUE = globals().get('MAX_DPO_PER_DATASET')
total_source_target = sum((spec.get('target_rows') or 0) for spec in all_specs if (spec.get('target_rows') or 0) > 0)

mixed_rows = []
dpo_mix_report = []

for spec in all_specs:
    path = DPO_DIR / f"{spec['name']}_dpo.jsonl"
    source_rows, source_report, source_bad_chunks = read_jsonl_objects(path)
    if source_bad_chunks:
        raise ValueError(f"Invalid DPO source JSONL: {path} -> {source_bad_chunks[0]}")

    if DPO_TOTAL_BUDGET_VALUE is None:
        dpo_target = spec.get('dpo_target_rows') or spec.get('target_rows') or spec.get('max_rows') or len(source_rows)
    elif total_source_target > 0:
        dpo_target = int(round(DPO_TOTAL_BUDGET_VALUE * (spec.get('target_rows') or 0) / total_source_target))
    else:
        dpo_target = DPO_TOTAL_BUDGET_VALUE

    if MAX_DPO_PER_DATASET_VALUE is not None:
        dpo_target = min(MAX_DPO_PER_DATASET_VALUE, dpo_target)
    if spec.get('max_rows') is not None:
        dpo_target = min(int(spec['max_rows']), dpo_target)

    sampled_rows = sample_records(source_rows, dpo_target, seed=RANDOM_SEED)
    mixed_rows.extend(sampled_rows)
    dpo_mix_report.append({
        'source_dataset': spec['name'],
        'source_rows': len(source_rows),
        'target_rows': dpo_target,
        'sampled_rows': len(sampled_rows),
        'sampling_mode': 'all_rows' if dpo_target is None or dpo_target >= len(source_rows) else 'sampled',
        'literal_separator_count': source_report['literal_separator_count'],
    })

write_jsonl_objects(DPO_MIXED_FILE, mixed_rows)
normalized_rows, normalized_report, normalized_bad_chunks = read_jsonl_objects(DPO_MIXED_FILE)
if normalized_bad_chunks:
    raise ValueError(f"Invalid mixed DPO JSONL after rewrite: {normalized_bad_chunks[0]}")

write_jsonl_objects(DPO_TRAIN_DIR / DPO_MIXED_FILE.name, normalized_rows)

print('[DPO mix normalization report]')
print(json.dumps({
    'dpo_total_budget': DPO_TOTAL_BUDGET_VALUE,
    'max_dpo_per_dataset': MAX_DPO_PER_DATASET_VALUE,
    'mixed_rows': len(normalized_rows),
    'literal_separator_count_in_mixed_file': normalized_report['literal_separator_count'],
    'sources': dpo_mix_report,
}, ensure_ascii=False, indent=2))


[DPO mix normalization report]
{
  "dpo_total_budget": null,
  "max_dpo_per_dataset": null,
  "mixed_rows": 10041,
  "literal_separator_count_in_mixed_file": 0,
  "sources": [
    {
      "source_dataset": "finqa_train",
      "source_rows": 3667,
      "target_rows": 3667,
      "sampled_rows": 3667,
      "sampling_mode": "all_rows",
      "literal_separator_count": 0
    },
    {
      "source_dataset": "convfinqa_train_turn",
      "source_rows": 6374,
      "target_rows": 6374,
      "sampled_rows": 6374,
      "sampling_mode": "all_rows",
      "literal_separator_count": 0
    }
  ]
}


In [ ]:
!python -m training.dpo_training --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged \
--tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--template_name qwen \
--validation_split_percentage 1 --eval_strategy no \
--train_file_dir /root/autodl-tmp/data/financial_reasoning_v3/dpo_pairs/train_dir \
--do_train --use_peft True \
--per_device_train_batch_size 1 --gradient_accumulation_steps 16 --gradient_checkpointing True \
--learning_rate 5e-6 --max_steps 100 --max_source_length 512 --max_target_length 256 \
--logging_steps 10 --save_steps 40  --logging_first_step True \
--target_modules q_proj,k_proj,v_proj,o_proj \
--lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
--torch_dtype bfloat16 --device_map auto \
--ddp_find_unused_parameters False \
--output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/dpo


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry dpo=/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged\
  --adapter_entry dpo=/root/autodl-tmp/outputs/financial_reasoning_v3/dpo \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/dpo_passk


1. 模型有没有学会“偏好 chosen > rejected”
2. 这种偏好学习是健康推进，还是过拟合/塌缩
3. 训练是否已经到该停的时候


最基础的是**判断训练有没有真正发生作用**。

这主要看 `train/loss` 和 `train/rewards/accuracies`：loss 是否持续下降、reward accuracy 是否从较低水平逐步上升。
- 如果这两个指标没有明显变化，说明模型几乎没有学到偏好；而如果 loss 快速下降、accuracy 很快接近 1，则说明模型已经很好地拟合了训练集中的 chosen/rejected 偏好关系，但这时反而需要警惕过拟合，因为 DPO 的目标是泛化偏好，而不是在训练集上“记住答案”。

在确认训练有效之后，第二步要关注模型到底 **“学了什么”** 。

这需要结合 `logps/chosen` 和 `logps/rejected` 来看。
- 理想情况是：模型对 chosen 的概率提高（logps 变得不那么负），同时对 rejected 的概率降低，这意味着模型既在强化正确回答，也在抑制错误回答。
- 但在很多实际训练中（包括你现在的曲线），后期往往表现为 chosen 基本不变，而 rejected 急剧下降，这说明模型主要是在“打压 rejected”，而不是进一步提升正向推理能力。
- 这种现象在对齐阶段非常常见，但对金融数值推理任务来说可能带来风险——模型更像是在模仿 preferred 模板，而不是更好地进行计算和推理。

第三步则是判断训练是否进入 **“过度优化”或“偏好饱和”** 阶段。

这主要依赖 `entropy`、`grad_norm` 和 `reward accuracy` 的联合判断：
1. 当 entropy 持续下降，说明模型输出分布越来越确定、越来越“自信”；
2. grad_norm 明显衰减，说明每一步参数更新的有效信息越来越少'；
3. reward accuracy 已接近 1。

这三者同时出现时，通常意味着模型已经把训练集偏好学完，后续训练收益极小，甚至可能开始损害泛化能力。
- 这也是为什么 DPO 训练往往需要“早停”，而不能单纯依赖 loss 最低来选择模型。

除此之外，还有一些辅助指标帮助你更细致理解训练过程。
- 例如 `mean_token_accuracy` 可以反映模型逐 token 的预测能力，但在 DPO 中它并不是核心指标，因为 DPO 优化的是排序而不是逐 token 拟合；
- `learning_rate` 帮助你解释 loss 或 grad_norm 的变化是否来自学习率调度；
- `epoch` 和 `num_tokens` 则用于判断数据是否被反复使用过多，从而导致过拟合风险。这些指标本身不直接决定模型好坏，但在解释训练行为时非常关键。

总结来说，DPO 曲线的分析核心不是“哪个指标更高或更低”，而是看这些指标之间是否形成一致的逻辑：是否有效学习了偏好、这种学习是正向增强还是负向打压、以及是否已经进入饱和甚至过拟合阶段。最终，任何训练曲线的结论都必须通过下游金融推理 benchmark 来验证，而不能仅凭训练指标判断模型优劣。


| 指标                       | 含义                        | 健康趋势    | 危险信号       | 主要作用       |
| ------------------------ | ------------------------- | ------- | ---------- | ---------- |
| train/loss               | 偏好优化目标（chosen > rejected） | 持续下降后趋稳 | 快速降到极低     | 判断训练是否有效   |
| rewards/accuracies       | chosen 是否优于 rejected 的比例  | 上升并接近高值 | 很快达到 1 并稳定 | 判断偏好是否已学满  |
| logps/chosen             | 对正确回答的概率                  | 逐渐上升    | 基本不变       | 判断正向增强     |
| logps/rejected           | 对错误回答的概率                  | 逐渐下降    | 过度下降       | 判断负向打压     |
| logits/chosen / rejected | 原始打分                      | 两者差距拉大  | 波动异常       | 辅助验证偏好分离   |
| entropy                  | 输出分布不确定性                  | 缓慢下降    | 持续大幅下降     | 判断模型是否变“僵” |
| grad_norm                | 参数更新幅度                    | 逐渐下降    | 过快衰减或爆炸    | 判断是否收敛或不稳定 |
| mean_token_accuracy      | token级预测准确率               | 稳定或略升   | 明显下降或无变化   | 判断是否提升生成能力 |
| learning_rate            | 学习率变化                     | 按计划变化   | 异常波动       | 辅助解释训练变化   |
| epoch                    | 数据遍历次数                    | 平稳增加    | 过高         | 判断是否重复训练过多 |
| num_tokens               | 累计训练token数                | 线性增长    | 无明显异常      | 衡量训练规模     |


> **DPO 的理想状态是：loss 下降、reward accuracy 提升、chosen 上升 + rejected 下降，同时 entropy 和 grad_norm 平稳衰减；一旦 reward accuracy≈1 且 entropy/grad_norm 同时塌缩，就需要考虑早停并转向 benchmark 验证。**


## 评估：SFT vs SFT+DPO

（当前主测 FinQA + ConvFinQA）

按照 `README_fingpt.md` 的 fix 计划，这里比较 `SFT` 和 `DPO`：
- 任务结果：`answer_accuracy`、`program_accuracy`、`numeric_parse_rate`
- 输出质量：`final_answer_coverage`、`program_section_coverage`、`structured_response_coverage`、`avg_prediction_chars`

评测设置：
- `base`：基座模型
- `sft`：`SFT` merge 后模型
- `dpo`：以 `SFT` merge 模型为 base，再挂载 `DPO` LoRA adapter
- `ConvFinQA`：优先 `dev_turn.json`
- `FinQA`：优先 `test.json`，不存在则回退 `dev.json`
